# Rally financiero 1 — análisis de series de tiempo
[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murua-Alvaro/clases-y-otras-tonteras/blob/rally-uno/rally_1/analisis_series_tiempo_colab.ipynb)

Corte fijo: **24 de septiembre de 2026**. Se estudia USD/MXN frente a S&P 500, VIX y Treasury 10Y. Los resultados se calculan al ejecutar; no hay cifras escritas manualmente.

In [ ]:
!pip -q install yfinance


In [ ]:
import os,random,warnings
warnings.filterwarnings("ignore")
SEED=42; os.environ["PYTHONHASHSEED"]=str(SEED); random.seed(SEED)
import numpy as np; np.random.seed(SEED)
import pandas as pd, matplotlib.pyplot as plt, yfinance as yf, statsmodels.api as sm
from sklearn.metrics import mean_absolute_error,mean_squared_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
print("Seed:",SEED,"| yfinance:",yf.__version__)


## Datos, transformaciones y control de calidad
Se conservan sólo fechas comunes de negociación; no se usa `ffill`, para evitar retornos cero artificiales cuando algún mercado estuvo cerrado. USD/MXN y S&P 500 se transforman a rendimientos logarítmicos porcentuales; VIX a cambio en puntos y ^TNX a cambio en puntos porcentuales.

In [ ]:
START="2020-01-01"; END="2026-09-25"
ticks=["MXN=X","^GSPC","^VIX","^TNX"]; names={"MXN=X":"USDMXN","^GSPC":"SP500","^VIX":"VIX","^TNX":"TNX10Y"}
raw=yf.download(ticks,start=START,end=END,auto_adjust=False,progress=False)
if raw.empty or not isinstance(raw.columns,pd.MultiIndex): raise RuntimeError("Descarga de Yahoo Finance inválida.")
p=raw["Close"].rename(columns=names)[["USDMXN","SP500","VIX","TNX10Y"]].dropna().sort_index()
assert p.index.max()<=pd.Timestamp("2026-09-24") and not p.isna().any().any()
d=pd.DataFrame(index=p.index)
d["usdmxn_ret"]=100*np.log(p.USDMXN).diff(); d["sp500_ret"]=100*np.log(p.SP500).diff()
d["vix_change"]=p.VIX.diff(); d["tnx_change"]=p.TNX10Y.diff(); d=d.replace([np.inf,-np.inf],np.nan).dropna()
print("Periodo:",p.index.min().date(),"a",p.index.max().date(),"| observaciones:",len(p))
display(d.describe().T,d.corr())

def adf(s,reg):
    r=adfuller(s.dropna(),regression=reg,autolag="AIC"); return r[0],r[1],r[2],r[3]
adf_tab=pd.DataFrame({
 "USD/MXN nivel":adf(p.USDMXN,"ct"),
 "USD/MXN rendimiento":adf(d.usdmxn_ret,"c"),
 "S&P 500 nivel":adf(p.SP500,"ct"),
 "S&P 500 rendimiento":adf(d.sp500_ret,"c")
},index=["ADF","p_value","rezagos","n_obs"]).T
display(adf_tab)

cut=int(len(d)*.8); tr,te=d.iloc[:cut].copy(),d.iloc[cut:].copy()
assert tr.index.max()<te.index.min()
ytr,yte=tr.usdmxn_ret,te.usdmxn_ret
def met(y,z): return {"MAE":float(mean_absolute_error(y,z)),"RMSE":float(mean_squared_error(y,z)**.5)}


## ARIMA y SARIMAX
ARIMA se selecciona por AIC entre `(p,0,q)` con `p,q=0..3`, **incluyendo (0,0,0)**. El SARIMAX usa factores externos del mismo día, así que su evaluación es **condicional/explicativa**, no un pronóstico ex ante puro de varios días.

In [ ]:
cand=[]
for a in range(4):
    for b in range(4):
        try:
            f=ARIMA(ytr,order=(a,0,b),trend="c").fit(); cand.append((a,b,f.aic))
        except Exception: pass
tab=pd.DataFrame(cand,columns=["p","q","AIC"]).sort_values("AIC").reset_index(drop=True)
if tab.empty: raise RuntimeError("No se pudo estimar ARIMA.")
bp,bq=int(tab.loc[0,"p"]),int(tab.loc[0,"q"])
ar=ARIMA(ytr,order=(bp,0,bq),trend="c").fit()
arp=pd.Series(ar.forecast(len(yte)).to_numpy(),index=yte.index)
x=["sp500_ret","vix_change","tnx_change"]
sx=SARIMAX(ytr,exog=tr[x],order=(bp,0,bq),trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
sxp=sx.get_forecast(len(yte),exog=te[x]).predicted_mean; sxp.index=yte.index
zero=pd.Series(0.0,index=yte.index)
cmp=pd.DataFrame([
 {"modelo":"Benchmark cero",**met(yte,zero)},
 {"modelo":f"ARIMA({bp},0,{bq})",**met(yte,arp)},
 {"modelo":"SARIMAX condicional",**met(yte,sxp)}
]).set_index("modelo")
print("ARIMA seleccionado:",(bp,0,bq)); display(tab.head(10),cmp)


## Sensibilidad del USD/MXN al mercado estadounidense
Se usa OLS con errores HAC/Newey-West. El coeficiente de S&P 500 es una **asociación parcial contemporánea**, no causalidad ni beta CAPM. La sensibilidad móvil de 60 sesiones es sólo descriptiva.

In [ ]:
r=d.copy(); r["lag_usdmxn"]=r.usdmxn_ret.shift(1); r=r.dropna()
X=sm.add_constant(r[["sp500_ret","vix_change","tnx_change","lag_usdmxn"]])
ols=sm.OLS(r.usdmxn_ret,X).fit(cov_type="HAC",cov_kwds={"maxlags":5})
print(ols.summary())
beta=float(ols.params.sp500_ret); pv=float(ols.pvalues.sp500_ret); ci=ols.conf_int().loc["sp500_ret"]
display(pd.DataFrame({"coef":[beta],"p_value":[pv],"IC95_inf":[ci.iloc[0]],"IC95_sup":[ci.iloc[1]]},index=["S&P 500"]))
if pv<.05 and beta<0: print("Asociación parcial negativa y significativa: caídas del S&P 500 tienden a coincidir con aumentos del USD/MXN.")
elif pv<.05: print("Asociación parcial positiva y significativa en esta especificación.")
else: print("El coeficiente parcial del S&P 500 no es significativo al 5%.")

rb=d.usdmxn_ret.rolling(60).cov(d.sp500_ret)/d.sp500_ret.rolling(60).var()
plt.figure(figsize=(12,4)); plt.plot(rb); plt.axhline(0,lw=.8); plt.title("Sensibilidad móvil USD/MXN vs S&P 500 — 60 sesiones"); plt.grid(alpha=.25); plt.show()


## Pronóstico y validaciones finales
El pronóstico final es univariado porque no supone valores futuros de S&P 500, VIX o Treasury. Las comprobaciones finales detectan errores de fechas, datos faltantes y resultados no finitos.

In [ ]:
fin=ARIMA(d.usdmxn_ret,order=(bp,0,bq),trend="c").fit()
fc=fin.get_forecast(5).summary_frame(alpha=.05)[["mean","mean_ci_lower","mean_ci_upper"]]
fc.columns=["pronostico_pct","limite_inferior_95","limite_superior_95"]; display(fc)

checks=pd.Series({
 "sin_NA_precios":not p.isna().any().any(),
 "sin_NA_modelo":not d.isna().any().any(),
 "corte_correcto":p.index.max()<=pd.Timestamp("2026-09-24"),
 "train_antes_test":tr.index.max()<te.index.min(),
 "metricas_finitas":np.isfinite(cmp[["MAE","RMSE"]].to_numpy()).all(),
 "beta_finita":np.isfinite(beta)},name="OK")
display(checks)
if not checks.all(): raise AssertionError("Falló una validación de integridad.")
print("Validaciones superadas.")
